In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('customer_logistics.csv')

# Check data types
print("Initial Data Types:")
print(df.dtypes)
print("\n" + "="*50)

def fix_data_types(df):

    df_clean = df.copy()
    
 
    date_columns = ['acquisition_date', 'order_date', 'payment_date']
    for col in date_columns:
        df_clean[col] = pd.to_datetime(df_clean[col])

    categorical_columns = ['customer_id', 'market_segment', 'supplier_id', 'order_id']
    for col in categorical_columns:
        df_clean[col] = df_clean[col].astype('category')
    
  
    numeric_columns = ['acquisition_cost_usd', 'order_value_usd', 'satisfaction_score', 
                       'support_tickets', 'lead_time_days']
    for col in numeric_columns:
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
    
    return df_clean

df = fix_data_types(df)
print("Data Types After Fixing:")
print(df.dtypes)


def analyze_missing_values(df):
  
    missing_data = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_Percentage': (df.isnull().sum() / len(df) * 100).round(2),
        'Data_Type': df.dtypes.values
    })
    
    missing_data = missing_data[missing_data['Missing_Count'] > 0]
    missing_data = missing_data.sort_values('Missing_Percentage', ascending=False)
    
    return missing_data

missing_analysis = analyze_missing_values(df)
print("Missing Values Analysis:")
print(missing_analysis)

# Listwise Deletion
def remove_missing_rows(df, threshold=0.05):
    df_dropped = df.dropna()
    
   
    missing_per_row = df.isnull().mean(axis=1)
    df_dropped_threshold = df[missing_per_row <= threshold]
    
    print(f"Original rows: {len(df)}")
    print(f"After dropping rows with any missing: {len(df_dropped)}")
    print(f"After dropping rows with >{threshold*100}% missing: {len(df_dropped_threshold)}")
    
    return df_dropped

# Mean/Median/Mode
def impute_missing_values(df):

    df_imputed = df.copy()
    

    numeric_cols = df.select_dtypes(include=[np.number]).columns
    for col in numeric_cols:
        if df_imputed[col].isnull().any():
            median_value = df_imputed[col].median()
            df_imputed[col] = df_imputed[col].fillna(median_value)
            print(f"Imputed {col} with median: {median_value}")

    categorical_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in categorical_cols:
        if df_imputed[col].isnull().any():
            mode_value = df_imputed[col].mode()[0]
            df_imputed[col] = df_imputed[col].fillna(mode_value)
            print(f"Imputed {col} with mode: {mode_value}")
    
    return df_imputed

# Time Series
def forward_backward_fill(df, column_name):

    df_filled = df.copy()
    df_filled[column_name] = df_filled[column_name].fillna(method='ffill')
    df_filled[column_name] = df_filled[column_name].fillna(method='bfill')
    
    return df_filled

# Duplicate Records
def detect_and_handle_duplicates(df):

  
    duplicate_count = df.duplicated().sum()
    print(f"Number of duplicate rows: {duplicate_count}")
    
    if duplicate_count > 0:
  
        duplicate_rows = df[df.duplicated(keep=False)]
        print(f"\nDuplicate records found:")
        print(duplicate_rows)

        df_unique = df.drop_duplicates(keep='first')
        print(f"\nRemoved {len(df) - len(df_unique)} duplicate rows")
        return df_unique
    
    return df

df = detect_and_handle_duplicates(df)


# Other detection
def visualize_outliers(df):

    numeric_cols = ['acquisition_cost_usd', 'order_value_usd', 'satisfaction_score', 
                    'support_tickets', 'lead_time_days']
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for idx, col in enumerate(numeric_cols):
        # Box plot
        df.boxplot(column=col, ax=axes[idx])
        axes[idx].set_title(f'Box Plot - {col}')
        axes[idx].set_ylabel(col)
        

        axes[idx].grid(True, alpha=0.3)
    
    axes[-1].axis('off')
    plt.tight_layout()
    plt.show()

visualize_outliers(df)
